In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd "/content/drive/MyDrive/Enterprise_AI_Platform"

/content/drive/MyDrive/Enterprise_AI_Platform


In [5]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/Enterprise_AI_Platform/src")

In [2]:
!python --version
!pwd

Python 3.12.13
/content/drive/MyDrive/Enterprise_AI_Platform


In [3]:
!python -m pip install -r requirements.txt

In [ ]:
!git init

Reinitialized existing Git repository in /content/drive/MyDrive/Enterprise_AI_Platform/.git/


In [ ]:
!git config user.name "dhammadipdk"
!git config user.email "dhammadipkamble3@gmail.com"

In [ ]:
!git status

^C


In [ ]:
!git branch

* main


In [ ]:
!git remote -v

origin	https://github.com/dhammadipdk/Enterprise_AI_Platform.git (fetch)
origin	https://github.com/dhammadipdk/Enterprise_AI_Platform.git (push)


In [ ]:
!PYTHONPATH=src python src/enterprise_ai_platform/main.py


╭─────── Startup ────────╮
│ Enterprise AI Platform │
│ Version 1.0.0          │
╰────────────────────────╯
Loading Configuration...
Initializing Logger...
Platform Bootstrap Complete.

Platform Ready.


In [ ]:
!git remote add origin https://github.com/dhammadipdk/Enterprise_AI_Platform.git

error: remote origin already exists.


In [ ]:
!git remote -v

origin	https://github.com/dhammadipdk/Enterprise_AI_Platform.git (fetch)
origin	https://github.com/dhammadipdk/Enterprise_AI_Platform.git (push)


In [ ]:
!git branch -M main

In [ ]:
!git add .

In [ ]:
!git commit -m "Sprint 1 - Task 2: Initial platform bootstrap"

On branch main
nothing to commit, working tree clean


In [ ]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
# Run this on root directory for tests..
# /content/drive/MyDrive/Enterprise_AI_Platform
# export PYTHONPATH=src
# pytest

In [ ]:
# git add .

# git commit -m "Sprint 1 - Task 3: Implement component and service registries"

# git push



### use this first if made changes in the github directly first
# git pull --rebase origin main

In [ ]:
## for ollama inference -> ./examples/model_LLM/model_engine_bootstrap.py
## Older colab cell
# !sudo apt-get install zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama server in the background
!nohup ollama serve > /content/ollama.log 2>&1 &

# Give it a couple seconds to start, then pull a small model
!sleep 3
!ollama pull llama3.2:3b

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
## Recommendation extraction try

from enterprise_ai_platform.domains.insurance.policy_advisor.recommendation_engine import PolicyRecommendationEngine

engine = PolicyRecommendationEngine(
    "src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv"
)
result = engine.recommend(
    vehicle_idv_rs=400000, vehicle_age_years=3, ncb_percent=20,
    coverage_priorities=["zero_dep", "roadside_assistance", "ncb_protect"],
    prefers_cashless=True, budget_cap_rs=12000,
)
print(result["recommendations"][0])
print(result["why_not_cheapest"])

{'policy_id': 'INS_C_FAMILY_CAR_PLUS', 'product_name': 'ClaimEase FamilyCar Plus', 'insurer_name': 'ClaimEase', 'coverage_type': 'comprehensive_plus', 'estimated_annual_premium_rs': 7926, 'suitability_score': 73.23, 'best_for': 'Family cars with moderate mileage and preference for smoother claims', 'exclusion_tags': 'Engine protect not included,Tyre wear excluded,RTI not included', 'plain_language_pitch': 'Balanced family-oriented cover with good service and add-on support.', 'matched_coverage': ['zero_dep', 'roadside_assistance', 'ncb_protect']}
ValueDrive ThirdParty Basic is cheaper (Rs 3673) but ranked lower: it matches 0 of 3 priority coverages you asked for, versus 3 for the top recommendation.


In [ ]:
## Ollama recommendation ... try

!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /content/ollama.log 2>&1 &

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import time
time.sleep(5)

In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [ ]:
import requests
print(requests.get("http://localhost:11434/api/tags").json())

{'models': [{'name': 'llama3.2:3b', 'model': 'llama3.2:3b', 'modified_at': '2026-07-27T13:05:30.052365957Z', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}]}


In [ ]:
from enterprise_ai_platform.model_engine import ModelService, OllamaAdapter, ModelDefinition, ProviderDefinition

model_service = ModelService()
model_service.register_provider(ProviderDefinition(name="ollama", description="Local Ollama"), OllamaAdapter())
model_service.register_model(ModelDefinition(
    name="explanation_model", version="1.0.0", provider="ollama",
    configuration={"ollama_model_name": "llama3.2:3b"},
))

response = model_service.execute("explanation_model", "Say hi in Hindi")
print(response.text)

नमस्ते (Namaste)


In [ ]:
from enterprise_ai_platform.tool_engine import ToolService
from enterprise_ai_platform.workflow_engine import WorkflowService
from enterprise_ai_platform.domains.insurance.policy_advisor import register_policy_advisor_workflow

tool_service = ToolService()
workflow_service = WorkflowService()
register_policy_advisor_workflow(
    workflow_service, tool_service, model_service,
    "src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv",
)

instance = workflow_service.execute("policy_advisor", initial_variables={"vehicle_idv_rs": 400000})

print("state:", instance.state)
print("error:", instance.error)
print("path:", [r.node_id for r in instance.node_history])
print("response_text:", instance.context.get_variable("response_text"))

state: ExecutionState.COMPLETED
error: None
path: ['start', 'check_slots', 'ask_clarifying_question', 'end_ask']
response_text: "आपके वाहन की उम्र कितनी है?" ("How old is your vehicle?")


In [ ]:
instance2 = workflow_service.execute("policy_advisor", initial_variables={
    "vehicle_idv_rs": 400000,
    "vehicle_age_years": 3,
    "ncb_percent": 20,
    "coverage_priorities": ["zero_dep", "roadside_assistance", "ncb_protect"],
    "prefers_cashless": True,
})

print("state:", instance2.state)
print("path:", [r.node_id for r in instance2.node_history])
print("response_text:", instance2.context.get_variable("response_text"))

state: ExecutionState.COMPLETED
path: ['start', 'check_slots', 'get_recommendations', 'format_explanation', 'end_recommend']
response_text: "Hi, hope u doing well! Wanna tell u about ClaimEase FamilyCar Plus policy. Annual premium is Rs 7926. This policy fits ur needs bcos it's balanced, family-oriented cover with good service & add-on support. Suitable for family cars with moderate mileage & preference for smooth claims. Note: ValueDrive ThirdParty Basic is cheaper (Rs 3673) but ranked lower in priority coverages u asked for - matches 0/3 vs 3 for top rec. Let me know if u interested"


In [ ]:
from pathlib import Path
from enterprise_ai_platform.knowledge_engine import KnowledgeService

knowledge_service = KnowledgeService()

knowledge_service.load_repository(
    "insurance",
    Path("src/enterprise_ai_platform/domains/insurance/knowledge"),
)

# THIS LINE WAS MISSING BEFORE -- load_repository only reads the folder
# structure, it never embeds anything or builds the search indexes.
# index_repository is the separate step that actually does that.
count = knowledge_service.index_repository("insurance")
print(f"Indexed {count} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 93 chunks


In [ ]:
matches = knowledge_service.hybrid_search(
    "insurance",
    "plain language recommendation exclusions disclosure",
    top_k=1,
    domain="regulatory_knowledge",
)
print(matches)
if matches:
    print(matches[0].chunk.metadata.get("implication_for_agent"))

[VectorStoreMatch(chunk=Chunk(content='knowledge_id: RK001\ntheme: Plain-language policy understanding\nsource_year: 2010-11\ninsurance_line: cross-line\nextracted_signal: IRDA emphasised full, plain, adequate and comparable product information and proposed a simple-language key features document.\nimplication_for_agent: Every recommendation should include a plain-language summary, premium details, charges, risks and what happens on discontinuance.\nmodel_use: retrieval, explanation, response quality guardrail\nsearch_tags: plain language,key features,transparency,mis-selling\nsource_file: Annual report 2010-11 Bi-lingual.pdf\nprovenance: extracted', repository='insurance', domain='regulatory_knowledge', asset='seed_data', chunk_index=0, metadata={'knowledge_id': 'RK001', 'theme': 'Plain-language policy understanding', 'source_year': '2010-11', 'insurance_line': 'cross-line', 'extracted_signal': 'IRDA emphasised full, plain, adequate and comparable product information and proposed a si

In [ ]:
from pathlib import Path
from enterprise_ai_platform.knowledge_engine import KnowledgeService

knowledge_service = KnowledgeService()
knowledge_service.load_repository(
    "insurance",
    Path("src/enterprise_ai_platform/domains/insurance/knowledge"),
)

register_policy_advisor_workflow(
    workflow_service, tool_service, model_service,
    "src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv",
    knowledge_service=knowledge_service,
    glossary_path="src/enterprise_ai_platform/domains/insurance/knowledge/jargon_glossary/entity_catalog.csv",
)

instance = workflow_service.execute("policy_advisor", initial_variables={
    "vehicle_idv_rs": 400000, "vehicle_age_years": 3, "ncb_percent": 20,
    "coverage_priorities": ["zero_dep", "roadside_assistance", "ncb_protect"],
    "prefers_cashless": True,
})
print(instance.context.get_variable("response_text"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

"Namaste! We understand why you're looking at our ClaimEase FamilyCar Plus policy. It's a balanced cover designed for families like yours, with good service and support too. If you have a family car with moderate mileage, this might be the best fit. One thing to keep in mind is that ValueDrive ThirdParty Basic is cheaper (Rs 3673), but it doesn't match all your priority needs like our top rec does - it matches 0 of 3 priorities versus 3 for our top rec. We also offer Zero Depreciation, Roadside Assistance and NCB Protection as add-ons to make life easier. Let's discuss how we can tailor this policy to suit your family's needs."


In [6]:
### Sprint 12.4 + 12.5 + 12.5 Change1 + .....(Top 3 Recommendation and Comparison)

In [6]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /content/ollama.log 2>&1 &

import time
time.sleep(5)

!ollama pull llama3.2:3b

!ollama list

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [6]:
import os

required_files = [
    "src/enterprise_ai_platform/domains/insurance/knowledge/regulatory_knowledge/seed_data.csv",
    "src/enterprise_ai_platform/domains/insurance/knowledge/complaint_playbooks/seed_data.csv",
    "src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv",
    "src/enterprise_ai_platform/domains/insurance/knowledge/jargon_glossary/entity_catalog.csv",
    "src/enterprise_ai_platform/domains/insurance/policy_advisor/recommendation_engine.py",
    "src/enterprise_ai_platform/domains/insurance/policy_advisor/tools.py",
    "src/enterprise_ai_platform/domains/insurance/policy_advisor/handlers.py",
    "src/enterprise_ai_platform/domains/insurance/policy_advisor/glossary.py",
    "src/enterprise_ai_platform/domains/insurance/policy_advisor/register_policy_advisor_workflow.py",
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("MISSING FILES -- fix these before continuing:")
    for f in missing:
        print(" -", f)
else:
    print("All required files present.")

All required files present.


In [7]:
from pathlib import Path

from enterprise_ai_platform.model_engine import (
    ModelService, OllamaAdapter, ModelDefinition, ProviderDefinition,
)
from enterprise_ai_platform.tool_engine import ToolService
from enterprise_ai_platform.workflow_engine import WorkflowService, ExecutionState
from enterprise_ai_platform.knowledge_engine import KnowledgeService
from enterprise_ai_platform.domains.insurance.policy_advisor import (
    register_policy_advisor_workflow,
)

CATALOG_PATH = "src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv"
GLOSSARY_PATH = "src/enterprise_ai_platform/domains/insurance/knowledge/jargon_glossary/entity_catalog.csv"
KNOWLEDGE_ROOT = Path("src/enterprise_ai_platform/domains/insurance/knowledge")

# 1. Model service, real Ollama
model_service = ModelService()
model_service.register_provider(
    ProviderDefinition(name="ollama", description="Local Ollama"), OllamaAdapter()
)
model_service.register_model(ModelDefinition(
    name="explanation_model", version="1.0.0", provider="ollama",
    configuration={"ollama_model_name": "llama3.2:3b"},
))

# 2. Tool service (Policy Advisor's tools get registered inside
#    register_policy_advisor_workflow below -- don't register them here too)
tool_service = ToolService()

# 3. Knowledge service -- load AND index (index_repository is the step
#    that was missing before; load_repository alone does nothing searchable)
knowledge_service = KnowledgeService()
knowledge_service.load_repository("insurance", KNOWLEDGE_ROOT)
indexed_count = knowledge_service.index_repository("insurance")
print(f"Indexed {indexed_count} chunks")
assert indexed_count > 0, "Indexing produced zero chunks -- check the knowledge_root path"

# 4. Workflow service, fully wired
workflow_service = WorkflowService()
register_policy_advisor_workflow(
    workflow_service, tool_service, model_service, CATALOG_PATH,
    knowledge_service=knowledge_service,
    glossary_path=GLOSSARY_PATH,
)

print("All services wired successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 93 chunks
All services wired successfully.


In [8]:
# Regulatory grounding
matches = knowledge_service.hybrid_search(
    "insurance", "plain language recommendation exclusions disclosure",
    top_k=1, domain="regulatory_knowledge",
)
assert matches, "Regulatory grounding returned nothing -- check indexing"
print("Regulatory note found:", matches[0].chunk.metadata.get("implication_for_agent"))

Regulatory note found: Every recommendation should include a plain-language summary, premium details, charges, risks and what happens on discontinuance.


In [9]:
# Jargon glossary
from enterprise_ai_platform.domains.insurance.policy_advisor.glossary import JargonGlossary

glossary = JargonGlossary(GLOSSARY_PATH)
zero_dep = glossary.lookup("zero_dep")
assert zero_dep is not None, "Glossary lookup failed for 'zero_dep'"
print("zero_dep definition:", zero_dep)
assert "epreciation" in zero_dep[1], "Definition should mention depreciation, not deductible"
print("Glossary is correct: mentions depreciation, not deductible.")

zero_dep definition: ('Zero Depreciation', 'In case of a claim, the insurer pays the full replacement cost of damaged parts without deducting depreciation for their age. This is NOT the same as a deductible.')
Glossary is correct: mentions depreciation, not deductible.


In [10]:
instance = workflow_service.execute("policy_advisor", initial_variables={
    "vehicle_idv_rs": 400000,
})

print("state:", instance.state)
print("error:", instance.error)
print("path:", [r.node_id for r in instance.node_history])
print("response_text:", instance.context.get_variable("response_text"))

assert instance.state == ExecutionState.COMPLETED
assert [r.node_id for r in instance.node_history] == [
    "start", "check_slots", "ask_clarifying_question", "end_ask",
]
print("\nBranch 1 PASSED.")

state: ExecutionState.COMPLETED
error: None
path: ['start', 'check_slots', 'ask_clarifying_question', 'end_ask']
response_text: "आपकी गाड़ी का वर्ष कितना है?" (Aapki gaadi ka varsh kitna hai?)

Branch 1 PASSED.


In [11]:
instance2 = workflow_service.execute("policy_advisor", initial_variables={
    "vehicle_idv_rs": 400000,
    "vehicle_age_years": 3,
    "ncb_percent": 20,
    "coverage_priorities": ["zero_dep", "roadside_assistance", "ncb_protect"],
    "prefers_cashless": True,
})

print("state:", instance2.state)
print("error:", instance2.error)
print("path:", [r.node_id for r in instance2.node_history])

recommendations_result = instance2.context.get_variable("recommendations_result")
print("\nNumber of recommendations returned:", len(recommendations_result["recommendations"]))
print("Number of pairwise comparisons:", len(recommendations_result["comparisons"]))

print("\n--- Final response_text ---")
print(instance2.context.get_variable("response_text"))

assert instance2.state == ExecutionState.COMPLETED
assert [r.node_id for r in instance2.node_history] == [
    "start", "check_slots", "get_recommendations", "format_explanation", "end_recommend",
]
assert len(recommendations_result["recommendations"]) == 3
assert len(recommendations_result["comparisons"]) == 3  # C(3,2) pairs
print("\nBranch 2 PASSED.")

state: ExecutionState.COMPLETED
error: None
path: ['start', 'check_slots', 'get_recommendations', 'format_explanation', 'end_recommend']

Number of recommendations returned: 3
Number of pairwise comparisons: 3

--- Final response_text ---
Arre, customer! 

Maine aayojit kiya hai aapke liye family car insurance options. Yahan par hain aapki sabse pasand policy:

1. **ClaimEase FamilyCar Plus** - Annual premium: Rs 7926
Yeh ek balanced cover hai jo parivaar ke liye taiyar hai, good service aur add-on support ka paalan karta hai. Yeh upyogkaron ke liye sabse achha hai jo apne family car mein moderate mileage hain aur smooth claims ki pasand kartey hain.

2. **ClaimEase ZeroDep Plus** - Annual premium: Rs 9455
Yeh ek high-protection plan hai jo nayi vehicles aur low claim deductions ke liye taiyar hai. Yeh upyogkartaon ke liye achha hai jo apne car mein naya hain ya finance kiya gaya hai aur low out-of-pocket claims ki pasand karte hain.

3. **UrbanShield FamilyCar Plus** - Annual premium:

In [12]:
from enterprise_ai_platform.domains.insurance.policy_advisor.recommendation_engine import PolicyRecommendationEngine

engine = PolicyRecommendationEngine("src/enterprise_ai_platform/domains/insurance/knowledge/policy_catalog/entity_catalog.csv")
result = engine.recommend_with_comparison(
    vehicle_idv_rs=400000, vehicle_age_years=3, ncb_percent=20,
    coverage_priorities=["zero_dep", "roadside_assistance", "ncb_protect"],
    prefers_cashless=True,
)
for c in result["comparisons"]:
    print(c["reasons"])

['ClaimEase FamilyCar Plus is Rs 1529 cheaper than ClaimEase ZeroDep Plus per year']
['ClaimEase FamilyCar Plus is Rs 181 cheaper than UrbanShield FamilyCar Plus per year']
['UrbanShield FamilyCar Plus is Rs 1348 cheaper than ClaimEase ZeroDep Plus per year']


In [8]:
### 1-vs-1 comparison added

instance = workflow_service.execute("policy_advisor", initial_variables={
    "vehicle_idv_rs": 400000, "vehicle_age_years": 3, "ncb_percent": 20,
    "policy_id_a": "INS_C_FAMILY_CAR_PLUS", "policy_id_b": "INS_C_ZERO_DEP_PLUS",
    "coverage_priorities": ["zero_dep", "roadside_assistance", "ncb_protect"],
    "prefers_cashless": True,
})
print("path:", [r.node_id for r in instance.node_history])
print(instance.context.get_variable("response_text"))

path: ['start', 'check_slots', 'get_comparison', 'format_comparison', 'end_compare']
"Namaste! We compared both policies and I'm happy to tell you that ClaimEase FamilyCar Plus is the better fit for you. Here's why:

Policy A (ClaimEase FamilyCar Plus) has an annual premium of exactly Rs 7926.

- You'll save Rs 1529 per year with this policy.
- The cashless garage score for A is 93, while B is 92 - that's a small difference!

Policy B (ClaimEase ZeroDep Plus) has an annual premium of exactly Rs 9455.

Here's the summary:
Policy A is recommended because it offers more savings (Rs 1529 per year) and a slightly higher cashless garage score. However, please note that we cannot provide detailed information on charges, discontinuance terms, or specific risks without explicit details from the insurer. If you need this information, I can request it for you."
